In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import time, random, os

In [2]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [3]:
if not os.path.exists("ECG5000_TRAIN.txt"):
    os.system('wget -q "https://www.timeseriesclassification.com/aeon-toolkit/ECG5000.zip"')
    os.system('unzip -q ECG5000.zip')

train_df = pd.read_csv("ECG5000_TRAIN.txt", header=None, sep=r'\s+')
test_df  = pd.read_csv("ECG5000_TEST.txt",  header=None, sep=r'\s+')

X_train_full = train_df.iloc[:, 1:].values.astype(np.float32)
y_train_full = train_df.iloc[:, 0].values
X_test_full  = test_df.iloc[:, 1:].values.astype(np.float32)
y_test_full  = test_df.iloc[:, 0].values

le = LabelEncoder()
y_train_full = le.fit_transform(y_train_full).astype(np.int64)
y_test_full  = le.transform(y_test_full).astype(np.int64)

def normalize_per_sample(X):
    mean = X.mean(axis=1, keepdims=True)
    std  = X.std(axis=1, keepdims=True) + 1e-8
    return (X - mean) / std

NUM_CLASSES = len(np.unique(y_train_full))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
class CNN_LSTM(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=7, padding=3), nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=5, padding=2), nn.ReLU(),
            nn.Conv1d(64, 128, kernel_size=3, padding=1), nn.ReLU(),
        )
        self.lstm = nn.LSTM(input_size=128, hidden_size=128, num_layers=2, batch_first=True)
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.cnn(x)
        x = x.permute(0, 2, 1)
        _, (h_n, _) = self.lstm(x)
        return self.classifier(h_n[-1])

In [5]:
def train_and_eval(seed, length, epochs=50, patience=10, lr=1e-3):
    set_seed(seed)

    X_norm = normalize_per_sample(X_train_full)
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_norm, y_train_full, test_size=0.15, stratify=y_train_full, random_state=seed
    )
    X_test_norm = normalize_per_sample(X_test_full)

    X_tr_t   = torch.tensor(X_tr[:, :length]).unsqueeze(1)
    X_val_t  = torch.tensor(X_val[:, :length]).unsqueeze(1)
    X_test_t = torch.tensor(X_test_norm[:, :length]).unsqueeze(1)
    y_tr_t, y_val_t, y_test_t = torch.tensor(y_tr), torch.tensor(y_val), torch.tensor(y_test_full)

    train_loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=32, shuffle=True)
    val_loader   = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=32, shuffle=False)
    test_loader  = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=32, shuffle=False)

    model = CNN_LSTM(NUM_CLASSES).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    best_val_acc, patience_counter = 0.0, 0
    best_state = None

    for epoch in range(epochs):
        model.train()
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(Xb), yb)
            loss.backward()
            optimizer.step()

        model.eval()
        preds, labels = [], []
        with torch.no_grad():
            for Xb, yb in val_loader:
                p = model(Xb.to(device)).argmax(dim=1).cpu().numpy()
                preds.extend(p); labels.extend(yb.numpy())
        val_acc = accuracy_score(labels, preds)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    model.load_state_dict(best_state)
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for Xb, yb in test_loader:
            p = model(Xb.to(device)).argmax(dim=1).cpu().numpy()
            preds.extend(p); labels.extend(yb.numpy())
    test_acc = accuracy_score(labels, preds)

    return test_acc

In [6]:
SEEDS = [0, 1, 7, 13, 21, 42, 99, 123, 256, 789]  # 10 seeds, more than before
results = []

for seed in SEEDS:
    acc_40  = train_and_eval(seed, length=40)
    acc_140 = train_and_eval(seed, length=140)
    print(f"Seed {seed:>4} | T=40: {acc_40:.4f} | T=140: {acc_140:.4f}")
    results.append({'seed': seed, 'acc_40': acc_40, 'acc_140': acc_140})

Seed    0 | T=40: 0.8620 | T=140: 0.9009
Seed    1 | T=40: 0.8418 | T=140: 0.8953
Seed    7 | T=40: 0.8493 | T=140: 0.9060
Seed   13 | T=40: 0.8467 | T=140: 0.9282
Seed   21 | T=40: 0.8627 | T=140: 0.9076
Seed   42 | T=40: 0.8418 | T=140: 0.9027
Seed   99 | T=40: 0.8624 | T=140: 0.8989
Seed  123 | T=40: 0.8267 | T=140: 0.8991
Seed  256 | T=40: 0.8547 | T=140: 0.8958
Seed  789 | T=40: 0.8667 | T=140: 0.8962


In [7]:
df = pd.DataFrame(results)
df.to_csv('lstm_t40_t140_multiseed.csv', index=False)

print(df)
print(f"\nT=40  | mean: {df['acc_40'].mean():.4f} | std: {df['acc_40'].std():.4f}")
print(f"T=140 | mean: {df['acc_140'].mean():.4f} | std: {df['acc_140'].std():.4f}")
print(f"\nT=40 consistently lower than T=140? {(df['acc_40'] < df['acc_140']).mean()*100:.0f}% of seeds")

   seed    acc_40   acc_140
0     0  0.862000  0.900889
1     1  0.841778  0.895333
2     7  0.849333  0.906000
3    13  0.846667  0.928222
4    21  0.862667  0.907556
5    42  0.841778  0.902667
6    99  0.862444  0.898889
7   123  0.826667  0.899111
8   256  0.854667  0.895778
9   789  0.866667  0.896222

T=40  | mean: 0.8515 | std: 0.0126
T=140 | mean: 0.9031 | std: 0.0098

T=40 consistently lower than T=140? 100% of seeds
